# NumPy 100 — Problems 71–80

Questions from [rougier/numpy-100](https://github.com/rougier/numpy-100) (MIT License). Solutions below are my own.

**Workflow:** try 10–15 min → docs → hints → official solution → rewrite from memory.

In [1]:
import numpy as np

---
### 71. Consider an array of dimension (5,5,3), how to multiply it by an array with dimensions (5,5)? (★★★)

In [2]:
rng = np.random.default_rng(42)
A = rng.random((5, 5, 3))
B = rng.random((5, 5))

try:
    A * B                               # shapes are aligned from the RIGHT: 3 vs 5 -> error
except ValueError as e:
    print("ValueError:", e)

C = A * B[:, :, None]                   # add a trailing axis: (5, 5) -> (5, 5, 1)
print(C.shape)                          # (5, 5, 3)
print(np.isclose(C[2, 3, 1], A[2, 3, 1] * B[2, 3]))   # True: spot-check one entry

ValueError: operands could not be broadcast together with shapes (5,5,3) (5,5) 
(5, 5, 3)
True


**Concept:** broadcasting lines shapes up starting from the *last* axis, so `(5, 5, 3)` and `(5, 5)` don't match. Insert a new axis with `B[:, :, None]` (or `B[..., np.newaxis]`) to make `B` shape `(5, 5, 1)`; the size-1 axis is then stretched across the 3 channels. This is how you scale each pixel of an image by a per-pixel factor.

**Needed help?** hint  
**Revisit on:** Nov 18

---
### 72. How to swap two rows of an array? (★★★)

In [3]:
A = np.arange(25).reshape(5, 5)
print(A[:2])

A[[0, 1]] = A[[1, 0]]                   # swap rows 0 and 1
print(A[:2])

# the "obvious" Python swap is WRONG for arrays, because rows are views:
B = np.arange(25).reshape(5, 5)
B[0], B[1] = B[1], B[0]
print(B[:2])                            # both rows are now the same

[[0 1 2 3 4]
 [5 6 7 8 9]]
[[5 6 7 8 9]
 [0 1 2 3 4]]
[[5 6 7 8 9]
 [5 6 7 8 9]]


**Concept:** `A[[0, 1]] = A[[1, 0]]` works because indexing with a *list* (fancy indexing) makes a copy of the right-hand side before assigning. The usual Python swap `B[0], B[1] = B[1], B[0]` fails: each `B[i]` is a *view*, so after the first assignment row 0 is overwritten and the second assignment copies the already-changed data.

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 73. Consider a set of 10 triplets describing 10 triangles (with shared vertices), find the set of unique line segments composing all the  triangles (★★★)

In [4]:
rng = np.random.default_rng(42)
faces = np.array([rng.choice(20, 3, replace=False) for _ in range(10)])   # 10 triangles of vertex ids

# each triangle contributes three edges: (v0,v1), (v1,v2), (v2,v0)
edges = np.concatenate([faces[:, [0, 1]],
                        faces[:, [1, 2]],
                        faces[:, [2, 0]]])      # shape (30, 2)

edges.sort(axis=1)                      # (a, b) and (b, a) are the same segment
segments = np.unique(edges, axis=0)     # drop duplicates (shared edges)

print(edges.shape, "->", segments.shape)
print(segments[:5])

(30, 2) -> (27, 2)
[[ 1  4]
 [ 1  8]
 [ 1 13]
 [ 1 14]
 [ 1 15]]


**Concept:** list the three edges of every triangle, sort the two vertices *within* each edge so `(a, b)` and `(b, a)` look identical, then `np.unique(..., axis=0)` removes duplicate rows. Triangles that share an edge produce the same row, so the unique count is smaller than 30. The original solution does this with `np.roll` and a structured view; this version is easier to read.

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 74. Given a sorted array C that corresponds to a bincount, how to produce an array A such that np.bincount(A) == C? (★★★)

In [5]:
C = np.bincount([1, 1, 2, 3, 4, 4, 6])
print(C)                                # [0 2 1 1 2 0 1]

A = np.repeat(np.arange(len(C)), C)     # value i is repeated C[i] times
print(A)                                # [1 1 2 3 4 4 6]
print(np.array_equal(np.bincount(A), C))   # True

[0 2 1 1 2 0 1]
[1 1 2 3 4 4 6]
True


**Concept:** `np.bincount` counts how often each integer appears, so the inverse is repeating each integer `i` exactly `C[i]` times: `np.repeat(np.arange(len(C)), C)`. The original order of the data is lost, but the counts match exactly. (If `A` should be unsorted, shuffle it afterwards.)

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 75. How to compute averages using a sliding window over an array? (★★★)

In [6]:
def moving_average(a, n=3):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]        # window sum = cumsum[i] - cumsum[i - n]
    return ret[n - 1:] / n

Z = np.arange(20)
m1 = moving_average(Z, n=3)
print(m1)

# two other ways, all giving the same result:
m2 = np.convolve(Z, np.ones(3) / 3, mode="valid")
m3 = np.lib.stride_tricks.sliding_window_view(Z, 3).mean(axis=-1)
print(np.allclose(m1, m2), np.allclose(m1, m3))     # True True

[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.]
True True


**Concept:** with a cumulative sum, each window sum is `cumsum[i] - cumsum[i - n]`, so the whole moving average takes one pass and no loop. Alternatives: `np.convolve(Z, np.ones(n)/n, mode="valid")`, or `sliding_window_view(Z, n).mean(axis=-1)`. The result is shorter than the input by `n - 1` elements.

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 76. Consider a one-dimensional array Z, build a two-dimensional array whose first row is (Z[0],Z[1],Z[2]) and each subsequent row is  shifted by 1 (last row should be (Z[-3],Z[-2],Z[-1]) (★★★)

In [7]:
from numpy.lib import stride_tricks

Z = np.arange(10)

# safe, modern way (NumPy 1.20+)
R = stride_tricks.sliding_window_view(Z, 3)
print(R)
print(R.shape)                          # (8, 3)

# the older, low-level way: a view with custom strides (no copy, but easy to misuse)
R2 = stride_tricks.as_strided(Z, shape=(len(Z) - 3 + 1, 3), strides=(Z.strides[0], Z.strides[0]))
print(np.array_equal(R, R2))            # True

[[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]
 [6 7 8]
 [7 8 9]]
(8, 3)
True


**Concept:** a sliding window can be built as a *view* of the same memory with no copying. `sliding_window_view(Z, 3)` does it safely; `as_strided` does it with manual shape and strides, but can read invalid memory if you get them wrong. The result is read-only (for the safe version), and rows overlap in memory, so don't write to it.

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 77. How to negate a boolean, or to change the sign of a float inplace? (★★★)

In [8]:
rng = np.random.default_rng(42)

# negate a boolean array in place
Z = rng.integers(0, 2, 8).astype(bool)
print(Z)
np.logical_not(Z, out=Z)
print(Z)

# change the sign of a float array in place
Y = rng.uniform(-1, 1, 5)
print(Y)
np.negative(Y, out=Y)
print(Y)

[False  True  True False False  True False  True]
[ True False False  True  True False  True False]
[-0.8116453   0.9512447   0.5222794   0.57212861 -0.74377273]
[ 0.8116453  -0.9512447  -0.5222794  -0.57212861  0.74377273]


**Concept:** ufuncs accept `out=`, so `np.logical_not(Z, out=Z)` flips a boolean array and `np.negative(Y, out=Y)` flips the sign of a float array without allocating a new array. `~Z` and `-Y` give the same values but create new arrays; `Y *= -1` is another in-place option for floats.

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 78. Consider 2 sets of points P0,P1 describing lines (2d) and a point p, how to compute distance from p to each line i (P0[i],P1[i])? (★★★)

In [9]:
def distance(P0, P1, p):
    # distance from point p to the infinite line through P0[i] and P1[i]
    T = P1 - P0                                         # direction of each line
    L = (T ** 2).sum(axis=1)                            # squared length of each direction
    U = -((P0[:, 0] - p[0]) * T[:, 0] + (P0[:, 1] - p[1]) * T[:, 1]) / L
    U = U.reshape(len(U), 1)                            # position of the projection along the line
    D = P0 + U * T - p                                  # vector from p to its projection
    return np.sqrt((D ** 2).sum(axis=1))

rng = np.random.default_rng(42)
P0 = rng.uniform(-10, 10, (10, 2))
P1 = rng.uniform(-10, 10, (10, 2))
p = rng.uniform(-10, 10, 2)

d = distance(P0, P1, p)
print(d)

# check with the cross-product formula: |T x (p - P0)| / |T|
T = P1 - P0
cross = T[:, 0] * (p[1] - P0[:, 1]) - T[:, 1] * (p[0] - P0[:, 0])
print(np.allclose(d, np.abs(cross) / np.hypot(T[:, 0], T[:, 1])))   # True

[8.07604907 8.65930014 3.27561857 6.36696813 5.28721025 2.01440467
 3.49597516 6.78856622 9.40290536 3.51723297]
True


**Concept:** project `p` onto each line: `U` says how far along the line the projection falls, `P0 + U*T` is that projection point, and the distance is the length of the vector from `p` to it. The same value comes from the cross-product formula `|T × (p − P0)| / |T|`. This measures distance to the *infinite* line; for a finite segment, clip `U` to `[0, 1]` first.

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 79. Consider 2 sets of points P0,P1 describing lines (2d) and a set of points P, how to compute distance from each point j (P[j]) to each line i (P0[i],P1[i])? (★★★)

In [10]:
def distances(P0, P1, P):
    # returns D with D[j, i] = distance from point P[j] to the line through P0[i], P1[i]
    T = P1 - P0                                                     # (n, 2)
    num = (T[None, :, 0] * (P[:, None, 1] - P0[None, :, 1])
         - T[None, :, 1] * (P[:, None, 0] - P0[None, :, 0]))        # (m, n) cross products
    return np.abs(num) / np.hypot(T[:, 0], T[:, 1])[None, :]

rng = np.random.default_rng(42)
P0 = rng.uniform(-10, 10, (10, 2))
P1 = rng.uniform(-10, 10, (10, 2))
P = rng.uniform(-10, 10, (5, 2))

D = distances(P0, P1, P)
print(D.shape)                          # (5, 10): 5 points x 10 lines

# check against the single-point function from problem 78, looped over the points
check = np.array([distance(P0, P1, p) for p in P])
print(np.allclose(D, check))            # True

(5, 10)
True


**Concept:** make the point axis and the line axis different with broadcasting: points become shape `(m, 1)` and lines `(1, n)`, so one expression gives all `m × n` distances with no Python loop. Compare with problem 78, which handles one point at a time. Check any vectorized version against the simple loop.

**Needed help?** solution  
**Revisit on:** Nov 18

---
### 80. Consider an arbitrary array, write a function that extracts a subpart with a fixed shape and centered on a given element (pad with a `fill` value when necessary) (★★★)

In [11]:
def subpart(Z, center, shape, fill=0):
    Z = np.asarray(Z)
    out = np.full(shape, fill, dtype=Z.dtype)
    z_slices, o_slices = [], []
    for c, s, n in zip(center, shape, Z.shape):
        start = c - s // 2                          # where the window begins in Z's coordinates
        z_lo, z_hi = max(start, 0), min(start + s, n)   # part of the window that lies inside Z
        size = max(z_hi - z_lo, 0)
        o_lo = z_lo - start                         # where that part lands inside the output
        z_slices.append(slice(z_lo, z_lo + size))
        o_slices.append(slice(o_lo, o_lo + size))
    out[tuple(o_slices)] = Z[tuple(z_slices)]
    return out

Z = np.arange(1, 26).reshape(5, 5)
print(subpart(Z, (1, 1), (3, 3)))               # fully inside
print(subpart(Z, (0, 0), (3, 3)))               # top-left corner, padded with 0
print(subpart(Z, (4, 4), (3, 3), fill=-1))      # bottom-right corner, padded with -1

# check against a simpler (but memory-hungry) pad-then-slice version
def subpart_pad(Z, center, shape, fill=0):
    pad = max(shape)
    P = np.pad(Z, pad, constant_values=fill)
    sl = tuple(slice(c + pad - s // 2, c + pad - s // 2 + s) for c, s in zip(center, shape))
    return P[sl]

ok = all(np.array_equal(subpart(Z, (i, j), (3, 3)), subpart_pad(Z, (i, j), (3, 3)))
         for i in range(5) for j in range(5))
print(ok)                                       # True

[[ 1  2  3]
 [ 6  7  8]
 [11 12 13]]
[[0 0 0]
 [0 1 2]
 [0 6 7]]
[[19 20 -1]
 [24 25 -1]
 [-1 -1 -1]]
True


**Concept:** create the output filled with `fill`, then copy only the part of the window that overlaps the array. For each axis, compute the window start (`center - size // 2`), clip it to the array bounds, and copy that slice into the matching position of the output. It works in any number of dimensions. The check compares against `np.pad`, which is simpler but copies the whole array.

**Needed help?** solution  
**Revisit on:** Nov 18